## Preprocessing

In [1]:
import nltk
import spacy
import re
import contractions

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
stop_words = stopwords.words("english")
lemmatizer = WordNetLemmatizer

def preprocess(text):
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r'W+', " ", text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

In [ ]:
df['clean_text'] = df['text'].apply(preprocess)

In [ ]:
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(df['clean_text'])

## NER

In [4]:
import spacy
nlp = spacy.load("en_core_web_sm")

content = ""

doc = nlp(content)

for ent in doc.ents:
    print(ent.text, ent.label_)

## text generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'gpt2'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = ' '

inputs = tokenizer(prompt, return_tensors='pt')

output = model.generate(
    **inputs,
    max_length = 50,
    do_sample=True,
    temperature=1.2,
    top_k = 50,
    top_p = 0.9
)

output = tokenizer.decoder(output, skip_special_tokens=True)

## LSTM for text generation

In [6]:
import torch
import torch.nn as nn

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

#preprocess text
text = ' '
words = text.split()
vocab = sorted(set(words))
stoi = {w:i for i,w in enumerate(vocab)}
vocab_size = len(vocab)

data = torch.tensor([stoi[w] for w in words])

seq_len = 10
X, Y = [], []

for i in range(len(data)-seq_len):
    X.append(data[i:i+seq_len])
    Y.append(data[i+1: seq_len+1])

X = torch.stack(X)
Y = torch.stack(Y)

loader = DataLoader(TensorDataset(X, Y), batch_size=32)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.embed = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 128)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out, _ = self.lstm(x)
        return self.fc(out)
    
model = LSTMModel()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for X,Y in loader:
    opt.zero_grad()
    out = model(x)
    loss = loss_fn(out, Y)
    loss.backward()
    opt.step()        

## classifier

In [8]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report

In [ ]:
nb_classifier = MultinomialNB()
svm_classifier = SVC(kernel='linear')

nb_classifier.fit(X_train, y_train)
svm_classifier.fit(X_train, y_train)

y_pred = nb_classifier.predict(X_test)

classification_report(y_test, y_pred)

## Topic Modelling

In [ ]:
import gensim
from gensim import corpora
from gensim.models import LdaModel

processed_docs = []

dictionary = corpora.dictionary(processed_docs)
corpus = []

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=2,
    passes=10
)

topics = lda_model.print_topics

## similarity

In [ ]:
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

tokenized_docs = []
model = Word2Vec(sentences=tokenized_docs, vector_size=100)

def doc_vector(doc):
    vector = []
doc1_vec = []
doc2_vec = []

similarity = cosine_similarity()

# search
def search(query):
    query_vec = []
    similarity = cosine_similarity()

    scores = similarity.flatten()
    ranked_docs = scores.argsort()[::-1]

## SA

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers

model = Sequential()
model.add(layers.Embedding(max_words, 40, input_length=max_len))
model.add(layers.Bidirectional(layers.LSTM(20, dropout=0.6)))
model.add(layers.Dense(3, activation='softmax'))

model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train, y_train,
                    epochs = 40,
                    validation_data=(),
                    )

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

class SentimentAnalyzer:
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()

    def analyze(self, text):
        scores = self.analyzer.polarity_scores(text)
        compound = scores['compound']
        sentiment_level = self.classify_sentiment(compound)

        return {
            "scores":scores,
            "compound":compound,
            "level":sentiment_level
        }
    
    def classify_sentiment(self, compound):
        pass
    